# Projeto Integrador de Mineração de Dados — Letterboxd

UEL · Mineração de Dados (2COP024) · 2º sem. 2026
**Integrantes:** Patrick Mendes Bisolato & Heitor Oliveira Nunes

## 1. Pergunta e base

**Pergunta:** com informações do filme disponíveis antes da estreia, dá para prever se ele vai ficar entre os **20% piores**, os **60% do meio** ou os **20% melhores** em nota média no Letterboxd?

**Alvo:** `classe` ∈ {ruim, medíocre, bom}, definida pelos percentis 20 e 80 da nota média (`rating`) **no treino**. Como as classes são ordinais, os modelos usam a versão numérica `y` (0 = ruim, 1 = medíocre, 2 = bom). Métricas: F1 macro, acurácia balanceada e kappa quadrático (classes desbalanceadas, 20/60/20).

**Base:** *Letterboxd (Movies Dataset)*, Kaggle, por gsimonx37 — https://www.kaggle.com/datasets/gsimonx37/letterboxd
~940 mil filmes em vários CSVs ligados por `id`; ~91 mil têm nota média, e só esses entram.


| Arquivo | Colunas | Uso |
|---|---|---|
| `movies.csv` | `id`, `name`, `date` (ano), `tagline`, `description`, `minute` (duração), `rating` | uma linha por filme; `rating` só define o alvo |
| `genres.csv` | `id`, `genre` | gêneros (0/1) |
| `themes.csv` | `id`, `theme` | 40 temas mais frequentes (0/1) |
| `countries.csv` | `id`, `country` | 15 países mais frequentes (0/1) e nº de países |
| `languages.csv` | `id`, `type`, `language` | 15 idiomas mais frequentes (0/1) e nº de idiomas |
| `studios.csv` | `id`, `studio` | 30 estúdios mais frequentes (0/1) |
| `actors.csv` | `id`, `name`, `role` | 200 atores mais frequentes (0/1) e tamanho do elenco |

**Fora do projeto:** pôsteres (imagens pesadas; não são baixados) e `releases.csv`.

In [ ]:
import kagglehub, pandas as pd

h = "gsimonx37/letterboxd"
arquivos = ["movies.csv", "genres.csv", "themes.csv", "countries.csv",
            "languages.csv", "studios.csv", "actors.csv"]

caminhos = {a: kagglehub.dataset_download(h, path=a) for a in arquivos}

movies = pd.read_csv(caminhos["movies.csv"])
genres = pd.read_csv(caminhos["genres.csv"])

In [ ]:
import pandas as pd, os
import kagglehub

print(movies.shape)
print(movies.dtypes)
print(movies.isna().mean().round(3))
movies.head()

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

SEED = 42

#Base com nota e ano
df = movies.dropna(subset=["rating", "date"]).copy()
df["date"] = df["date"].astype(int)
df = df[df["date"] <= 2026].drop_duplicates("id").reset_index(drop=True)
print("filmes com nota e ano:", len(df))

#divisão unica treino/teste, feita ANTES de definir o alvo.

faixa = pd.qcut(df["rating"], 10, labels=False, duplicates="drop")
tr_idx, te_idx = train_test_split(df.index, test_size=0.2, stratify=faixa, random_state=SEED)

#alvo em 3 classes, cortes (percentis 20 e 80) calculados só no treino
q20, q80 = df.loc[tr_idx, "rating"].quantile([0.20, 0.80])
print(f"cortes (treino): ruim <= {q20:.2f} | bom >= {q80:.2f}")
df["classe"] = np.select([df["rating"] <= q20, df["rating"] >= q80],
                         ["ruim", "bom"], default="mediocre")
ordem = {"ruim": 0, "mediocre": 1, "bom": 2}
df["y"] = df["classe"].map(ordem)          #versão ordinal do alvo usada nos modelos

treino, teste = df.loc[tr_idx], df.loc[te_idx]
print(len(treino), len(teste))
print(pd.DataFrame({"treino": treino["classe"].value_counts(normalize=True),
                    "teste": teste["classe"].value_counts(normalize=True)}).round(3))

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

# movies e genres já foram carregados na primeira célula (download seletivo, sem posters)

print(movies.columns.tolist()); print(genres.columns.tolist())
print(genres.head())

# 1) Base com nota e ano
df = movies.dropna(subset=["rating", "date"]).copy()
df["date"] = df["date"].astype(int)
df = df[df["date"] <= 2026]
print("filmes com nota e ano:", len(df))

# 2) Alvo em 3 classes por percentil (20% / 60% / 20%)
q20, q80 = df["rating"].quantile([0.20, 0.80])
print(f"cortes: ruim <= {q20:.2f} | bom >= {q80:.2f}")
df["classe"] = np.select(
    [df["rating"] <= q20, df["rating"] >= q80],
    ["ruim", "bom"], default="mediocre")
print(df["classe"].value_counts(normalize=True).round(3))

# 3) Split ANTES de qualquer imputação, escala ou seleção
treino, teste = train_test_split(
    df, test_size=0.2, stratify=df["classe"], random_state=42)
print(len(treino), len(teste))

In [ ]:
# Numéricas
print(treino.groupby("classe")[["minute", "date", "rating"]].agg(["mean", "median"]).round(2))

# Gêneros: proporção de filmes de cada classe que têm cada gênero
# (ajuste os nomes das colunas de genres.csv conforme o head())
g = genres.merge(treino[["id", "classe"]], on="id")
tabela = pd.crosstab(g["genre"], g["classe"], normalize="columns") * 100
tabela["dif_bom_ruim"] = tabela["bom"] - tabela["ruim"]
print(tabela.sort_values("dif_bom_ruim").round(1))

In [ ]:
#pergunta inversa
g = genres.merge(treino[["id", "classe"]], on="id")
p = pd.crosstab(g["genre"], g["classe"], normalize="index") * 100
p["n"] = g["genre"].value_counts()
print(p.sort_values("ruim", ascending=False).round(1))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

# Gêneros em colunas 0/1 (sem aprender nada dos dados, então pode ser antes do split)
gen = pd.crosstab(genres["id"], genres["genre"]).clip(upper=1)
gen.columns = ["g_" + c for c in gen.columns]
df2 = df.join(gen, on="id")   # join mantém o índice de df (necessário para tr_idx/te_idx)
gcols = [c for c in df2.columns if c.startswith("g_")]
df2[gcols] = df2[gcols].fillna(0)

X = df2[["minute", "date"] + gcols]      
X_tr, X_te = X.loc[tr_idx], X.loc[te_idx]   
y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]

pre = ColumnTransformer([
    ("minute", Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("log", FunctionTransformer(np.log1p)),
                         ("sc", StandardScaler())]), ["minute"]),
    ("ano", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), ["date"]),
    ("gen", "passthrough", gcols),
])
print(X_tr.shape, X_te.shape, y_tr.value_counts(normalize=True).round(3).to_dict())

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, cohen_kappa_score

# Alvo já é ordinal (0 = ruim, 1 = medíocre, 2 = bom); a ordem importa para o kappa
y_tr_o, y_te_o = y_tr, y_te

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "f1_macro": "f1_macro",
    "bal_acc": "balanced_accuracy",
    "kappa_q": make_scorer(cohen_kappa_score, weights="quadratic"),
}

modelos = {
    "logistica": (
        Pipeline([("pre", pre),
                  ("clf", LogisticRegression(class_weight="balanced", max_iter=2000))]),
        {"clf__C": [0.1, 1, 10]},
    ),
    "floresta": (
        Pipeline([("pre", pre),
                  ("clf", RandomForestClassifier(class_weight="balanced",
                                                 n_estimators=200, n_jobs=-1,
                                                 random_state=42))]),
        {"clf__max_depth": [8, 16, None], "clf__min_samples_leaf": [5, 20]},
    ),
}

melhores = {}
for nome, (pipe, grade) in modelos.items():
    gs = GridSearchCV(pipe, grade, scoring="f1_macro", cv=cv, n_jobs=-1)
    gs.fit(X_tr, y_tr_o)
    melhores[nome] = gs.best_estimator_
    print(f"\n{nome}: melhores parâmetros {gs.best_params_}")

    # Mede o modelo escolhido com as 3 métricas, também por validação cruzada
    r = cross_validate(gs.best_estimator_, X_tr, y_tr_o, cv=cv, scoring=scoring)
    for m in scoring:
        v = r[f"test_{m}"]
        print(f"  {m}: {v.mean():.3f} ± {v.std():.3f}")

In [ ]:
# Estrutura dos arquivos extras
for a in ["themes.csv", "countries.csv", "languages.csv", "studios.csv", "actors.csv"]:
    d = pd.read_csv(caminhos[a], nrows=3)
    print(a, d.columns.tolist())
    print(d.head(3), "\n")

In [ ]:
def ler(arq):
    return pd.read_csv(caminhos[arq])

def escolher_coluna(d, preferidas):
    """Coluna com o valor (tema, país...). Tenta os nomes esperados; senão a de texto com mais valores distintos."""
    for p in preferidas:
        if p in d.columns:
            return p
    cand = [c for c in d.columns if c != "id" and d[c].dtype == object]
    return max(cand, key=lambda c: d[c].nunique())

def multi_hot(arq, preferidas, top, prefixo):
    d = ler(arq)
    col = escolher_coluna(d, preferidas)
    d = d.dropna(subset=[col]).drop_duplicates(["id", col])
    topn = d[col].value_counts().head(top).index      # só frequência, sem olhar o alvo
    d = d[d[col].isin(topn)]
    m = pd.crosstab(d["id"], d[col]).clip(upper=1)
    m.columns = [f"{prefixo}_{c}" for c in m.columns]
    print(f"{arq}: coluna usada = '{col}', {m.shape[1]} colunas 0/1")
    return m

def contagem(arq, nome):
    return ler(arq).groupby("id").size().rename(nome)

extras = [
    multi_hot("themes.csv",    ["theme"],    top=40, prefixo="tema"),
    multi_hot("countries.csv", ["country"],  top=15, prefixo="pais"),
    multi_hot("languages.csv", ["language"], top=15, prefixo="idioma"),
    multi_hot("studios.csv",   ["studio"],   top=30, prefixo="estudio"),
    multi_hot("actors.csv",    ["name"],     top=200, prefixo="ator"),
    contagem("countries.csv", "n_paises"),
    contagem("languages.csv", "n_idiomas"),
    contagem("actors.csv",    "n_atores"),
]

df3 = df2.join(pd.concat(extras, axis=1), on="id")

bin_cols = [c for c in df3.columns if c.startswith(("g_", "tema_", "pais_", "idioma_", "estudio_", "ator_"))]
cnt_cols = ["n_paises", "n_idiomas", "n_atores"]
df3[bin_cols] = df3[bin_cols].fillna(0)
df3[cnt_cols] = df3[cnt_cols].fillna(0)   # 0 = sem registro (não necessariamente "sem elenco")

# texto: sinopse + tagline (faltantes viram vazio)
df3["texto"] = (df3["description"].fillna("") + " " + df3["tagline"].fillna("")).str.strip()

print(df3.shape, "| filmes com texto:", (df3["texto"] != "").mean().round(3))

In [ ]:
#Ablação por blocos A e D
#Mesmo modelo, mesma validação cruzada, só no treino.
#Se o elenco (C) não melhora em relação a B,
#ele acrescenta pouco depois de conhecer o resto.
#Se a sinopse (D) não melhora, o texto não acrescenta nada.

from sklearn.feature_extraction.text import TfidfVectorizer

N_TREES = 100        # suba para 200 se o tempo permitir
MAX_TFIDF = 2000     # termos do TF-IDF

y4 = df3["y"] 

grupos = {
    "gen":    dict(bin=[c for c in bin_cols if c.startswith("g_")],  cnt=[]),
    "ctx":    dict(bin=[c for c in bin_cols if c.startswith(("tema_", "pais_", "idioma_", "estudio_"))],
                   cnt=["n_paises", "n_idiomas"]),
    "elenco": dict(bin=[c for c in bin_cols if c.startswith("ator_")], cnt=["n_atores"]),
}

def montar(blocos, usar_texto, clf):
    cnt = ["minute"] + sum((grupos[b]["cnt"] for b in blocos), [])
    binar = sum((grupos[b]["bin"] for b in blocos), [])
    etapas = [
        ("cnt", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("log", FunctionTransformer(np.log1p)),
                          ("sc", StandardScaler())]), cnt),
        ("ano", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), ["date"]),
        ("bin", "passthrough", binar),
    ]
    if usar_texto:
        etapas.append(("txt", TfidfVectorizer(max_features=MAX_TFIDF, min_df=5,
                                              stop_words="english", ngram_range=(1, 2)), "texto"))
    colunas = ["date"] + cnt + binar + (["texto"] if usar_texto else [])
    return Pipeline([("pre", ColumnTransformer(etapas)), ("clf", clf)]), colunas

def rf():
    return RandomForestClassifier(class_weight="balanced", n_estimators=N_TREES,
                                  min_samples_leaf=5, n_jobs=-1, random_state=42)

versoes = {
    "A: básico":     (["gen"], False),
    "B: + contexto": (["gen", "ctx"], False),
    "C: + elenco":   (["gen", "ctx", "elenco"], False),
    "D: + sinopse":  (["gen", "ctx", "elenco"], True),
}

linhas = []
for nome, (blocos, txt) in versoes.items():
    pipe, colunas = montar(blocos, txt, rf())
    r = cross_validate(pipe, df3.loc[tr_idx, colunas], y4[tr_idx], cv=cv,
                       scoring={"f1_macro": "f1_macro",
                                "kappa_q": make_scorer(cohen_kappa_score, weights="quadratic")})
    linhas.append({"versão": nome, "n_atributos": len(colunas) - 1,
                   "F1 macro": r["test_f1_macro"].mean(), "± F1": r["test_f1_macro"].std(),
                   "kappa quad.": r["test_kappa_q"].mean()})
    print(nome, "F1 macro: %.3f ± %.3f" % (r["test_f1_macro"].mean(), r["test_f1_macro"].std()))

res_ablacao = pd.DataFrame(linhas).set_index("versão").round(3)
res_ablacao

In [ ]:
#Leitura de texto: quais palavras puxam para "ruim" e para "bom"

txt_pipe = Pipeline([
    ("tf", TfidfVectorizer(max_features=5000, min_df=5, stop_words="english")),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
txt_pipe.fit(df3.loc[tr_idx, "texto"], y4[tr_idx])

termos = np.array(txt_pipe.named_steps["tf"].get_feature_names_out())
coef = txt_pipe.named_steps["lr"].coef_          # linhas: 0=ruim, 1=mediocre, 2=bom
for i, nome in enumerate(["ruim", "mediocre", "bom"]):
    print(f"\n{nome}:", ", ".join(termos[np.argsort(coef[i])[::-1][:15]]))

f1_txt = cross_validate(txt_pipe, df3.loc[tr_idx, "texto"], y4[tr_idx], cv=cv, scoring="f1_macro")["test_score"]
print("\nF1 macro só com texto: %.3f ± %.3f" % (f1_txt.mean(), f1_txt.std()))